## Analyzing Hidden States based on n-grams (Amazon Polarity Dataset)

### The three splits

Amazon Polarity originally provides:
- Official train set: 3,600,000 reviews
- Official test set:    400,000 reviews

We split the official training set using: DATA_SPLIT_SEED = 1 into:
train_data:       3,550,000 reviews
validation_data:     50,000 reviews
test_data:          400,000 reviews

**train_data**

Used to update the model weights during fine-tuning.
train_data = dataset["train"]

**validation_data**

Taken out of the original training set before fine-tuning. It is used to measure performance during development, but it is not used for gradient updates.
validation_data = dataset["validation"]
Because train_test_split() creates two disjoint partitions, these 50,000 samples are not in train_data.

**test_data**

This is the unchanged official Amazon Polarity test set:
test_data = dataset["test"]
It was not part of the 3.6-million-example training pool and was not used to train the model.


**Predictions are aligned**

The .npz contains predictions for the official test set in its original order:
amazon_polarity_test_with_preds.npz

The notebook now loads the same:
test_data = dataset["test"]

It checks:
len(test_data) == len(test_preds) == 400_000
and also compares the labels and texts. Therefore:
test_preds[i] corresponds to test_data[i]

**What the seeds affects**

DATA_SPLIT_SEED = 1
determines which 50,000 examples are selected as validation. It does not affect the official test set.

MODEL_SEED = 1
controls model-training randomness.

For seeds 1, 2, and 42, keep:
DATA_SPLIT_SEED = 1
and change only MODEL_SEED.

---
## Setting up S1 Vocabulary

Imports

In [21]:
from pathlib import Path
from collections import Counter, defaultdict
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import ClassLabel, Dataset, DatasetDict, load_dataset, load_from_disk
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.preprocessing import TransactionEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import normalize
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import BertForSequenceClassification, BertModel, BertTokenizerFast


Load models and tokenizer: fine-tuned Amazon Polarity BERT and pretrained BERT

In [22]:
# Change only MODEL_SEED when analysing another fine-tuned model.
MODEL_SEED = 1

model_path = Path(f"./polarity_seed_{MODEL_SEED}/model")

if not model_path.exists():
    raise FileNotFoundError(
        f"Fine-tuned model not found at: {model_path.resolve()}"
    )

tokenizer = BertTokenizerFast.from_pretrained(str(model_path))
ft_bert = BertForSequenceClassification.from_pretrained(str(model_path))

pretrained_name = "bert-base-uncased"
pt_bert = BertModel.from_pretrained(pretrained_name)

print("Model seed:", MODEL_SEED)
print("Fine-tuned model:", model_path.resolve())


Model seed: 1
Fine-tuned model: /Users/leosteiner/Desktop/transformer-shortcut-mitigation/polarity_seed_1/model


Load Dataset

In [23]:
# DATA_SPLIT_SEED stays fixed so all models use the same train/validation split.
DATA_SPLIT_SEED = 1
VALIDATION_SIZE = 50_000

# Shared cache used by seed 1, seed 2, and seed 42 models.
# It is deliberately outside the polarity_seed_* model folders.
PROCESSED_DATASET_PATH = Path(
    f"./amazon_polarity_processed_split_seed_{DATA_SPLIT_SEED}"
)


def create_text_column(batch):
    """Combine title and review body into one text field."""
    texts = []

    for title, content in zip(batch["title"], batch["content"]):
        title = "" if title is None else str(title).strip()
        content = "" if content is None else str(content).strip()

        text = " ".join(
            part for part in (title, content) if part
        )
        texts.append(text)

    return {"text": texts}


if PROCESSED_DATASET_PATH.exists():
    print("Loading cached processed Amazon Polarity dataset...")
    dataset = load_from_disk(str(PROCESSED_DATASET_PATH))

else:
    print("No processed cache found.")
    print("Downloading and preprocessing Amazon Polarity once...")

    raw_dataset = load_dataset("fancyzhx/amazon_polarity")
    raw_train = raw_dataset["train"]

    # Stratified splitting requires a ClassLabel feature.
    if not isinstance(raw_train.features["label"], ClassLabel):
        raw_train = raw_train.class_encode_column("label")

    train_validation_split = raw_train.train_test_split(
        test_size=VALIDATION_SIZE,
        stratify_by_column="label",
        seed=DATA_SPLIT_SEED,
    )

    dataset = DatasetDict({
        "train": train_validation_split["train"],
        "validation": train_validation_split["test"],
        "test": raw_dataset["test"],
    })

    # After this mapping, every split contains only text and label.
    dataset = dataset.map(
        create_text_column,
        batched=True,
        remove_columns=["title", "content"],
        desc="Creating combined review text",
    )

    dataset.save_to_disk(str(PROCESSED_DATASET_PATH))
    print("Processed dataset saved to:")
    print(PROCESSED_DATASET_PATH.resolve())


# Safety checks against accidentally loading an incompatible cache.
expected_splits = {"train", "validation", "test"}
if set(dataset.keys()) != expected_splits:
    raise ValueError(f"Unexpected dataset splits: {dataset.keys()}")

for split_name in ("train", "validation", "test"):
    if set(dataset[split_name].column_names) != {"text", "label"}:
        raise ValueError(
            f"Unexpected columns in {split_name}: "
            f"{dataset[split_name].column_names}"
        )

print(dataset)
print("Data split seed:", DATA_SPLIT_SEED)
print("Processed dataset path:", PROCESSED_DATASET_PATH.resolve())


Loading cached processed Amazon Polarity dataset...
DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 3550000
    })
    validation: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 400000
    })
})
Data split seed: 1
Processed dataset path: /Users/leosteiner/Desktop/transformer-shortcut-mitigation/amazon_polarity_processed_split_seed_1


In [24]:
train_data = dataset["train"]
validation_data = dataset["validation"]
official_test_data = dataset["test"]

# This notebook reproduces the test-set shortcut analysis used in the other notebooks.
pipeline_data = official_test_data
test_data = official_test_data

assert len(train_data) == 3_550_000
assert len(validation_data) == 50_000
assert len(test_data) == 400_000

print("Training examples:", len(train_data))
print("Validation examples:", len(validation_data))
print("Pipeline/test examples:", len(test_data))


Training examples: 3550000
Validation examples: 50000
Pipeline/test examples: 400000


## Set up device

In [25]:
# Device setup
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

pt_bert = pt_bert.to(device)
ft_bert = ft_bert.to(device)

pt_bert.eval()
ft_bert.eval()

print("Device:", device)


Device: mps


Method used to extract Saliency.

Source: [Understanding Text Classification Data and Models Using Aggregated Input Salience](https://arxiv.org/abs/2211.05485)

In [26]:
def grad_l2_saliency(
    text: str,
    model,
    tokenizer,
    device,
    target_class: int | None = None,
    max_length: int = 256
):
    """
    Compute Grad-L2 saliency for one input text.

    Returns:
        tokens: list[str]          # wordpiece tokens incl. special tokens
        saliency: np.ndarray       # shape (seq_len,)
        input_ids: np.ndarray      # token ids (same length as saliency)
    """
    model.eval()

    # 1) Tokenize
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )
    input_ids = enc["input_ids"].to(device)           # [1, L] L = sequence length
    attention_mask = enc["attention_mask"].to(device) # [1, L]
    token_type_ids = enc.get("token_type_ids")
    if token_type_ids is not None:
        token_type_ids = token_type_ids.to(device)

    # 2) Get word embeddings and make them a leaf tensor with grad
    with torch.no_grad():
        word_embeds = model.bert.embeddings.word_embeddings(input_ids)
    word_embeds = word_embeds.detach().requires_grad_(True)  # [1, L, H]

    # 3) Forward pass using inputs_embeds so we can backprop to embeddings
    model.zero_grad()
    outputs = model(
        inputs_embeds=word_embeds, #pass embeddings instead of input_ids
        attention_mask=attention_mask,
        token_type_ids=token_type_ids
    )
    logits = outputs.logits  # [1, num_labels]

    # 4) Choose the logit to explain: pick predicted label by default
    if target_class is None:
        target_class = logits.argmax(dim=-1).item()  # predicted label

    # using the predicted class as scalar for autograd
    score = logits[0, target_class] # standard tensor indexing: tensor[row_index, column_index]

    # 5) Backward: d score / d word_embeds # computes gradients of score with respect to all tensors that have requires_grad=True in the computation graph.
    score.backward()

    grads = word_embeds.grad  # [1, L, H] store gradient 

    # 6) L2 norm over hidden dimension -> one score per token
    saliency = torch.norm(grads, p=2, dim=-1).squeeze(0)  # [1,L,H ] -> [1,L] -> [L] collapsing H

    # 7) Move to CPU / numpy
    saliency = saliency.detach().cpu().numpy()
    input_ids_cpu = input_ids.squeeze(0).detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids_cpu)

    return tokens, saliency, input_ids_cpu


Predict Test Set in preparation for S1 Vocabulary

In [27]:
def predict_labels(
    dataset_split,
    model,
    tokenizer,
    device,
    batch_size=32,
    max_length=256,
):
    model.eval()
    loader = DataLoader(
        dataset_split,
        batch_size=batch_size,
        shuffle=False,
    )

    all_preds = []

    with torch.no_grad():
        for batch in loader:
            texts = batch["text"]

            enc = tokenizer(
                list(texts),
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max_length,
            ).to(device)

            logits = model(**enc).logits
            preds = logits.argmax(dim=-1).cpu()
            all_preds.extend(preds.tolist())

    return np.array(all_preds)


npz_path = "amazon_polarity_test_with_preds.npz"
npy_path = "test_preds.npy"

if os.path.exists(npz_path):
    saved_data = np.load(npz_path, allow_pickle=True)
    test_preds = saved_data["pred"]

    # Confirm that predictions and the loaded official test split are aligned.
    assert len(test_data) == len(test_preds) == 400_000
    assert np.array_equal(np.asarray(test_data["label"]), saved_data["label"])

    if "text" in saved_data.files:
        chunk_size = 10_000
        for start in range(0, len(test_data), chunk_size):
            end = min(start + chunk_size, len(test_data))
            dataset_text = np.asarray(test_data[start:end]["text"], dtype=object)
            saved_text = saved_data["text"][start:end]
            assert np.array_equal(dataset_text, saved_text)

    print("Loaded aligned predictions from:", npz_path)

elif os.path.exists(npy_path):
    test_preds = np.load(npy_path)
    assert len(test_data) == len(test_preds) == 400_000
    print("Loaded predictions from:", npy_path)

else:
    test_preds = predict_labels(
        test_data,
        ft_bert,
        tokenizer,
        device,
        max_length=256,
    )

    np.save(npy_path, test_preds)
    print("Generated and saved predictions to:", npy_path)

print("Number of predictions:", len(test_preds))


Loaded aligned predictions from: amazon_polarity_test_with_preds.npz
Number of predictions: 400000


Function to extract top 5 salient tokens per sentence.

In [28]:
def topk_vocab_tokens_with_maxpos(input_ids, saliency, top_k=5, exclude_special_ids=None):
    """
    For one sentence:
    - For each vocab token_id, keep the maximum |saliency| value observed in this sentence,
      and the token position where that maximum occurs.
    - Return the top_k token_ids ranked by that max |saliency|.

    Output format (length <= top_k):
      [{"token_id": int, "position": int, "saliency": float}, ...]
    """
    exclude_special_ids = set(exclude_special_ids or [])
    best = {}  # token_id -> (best_abs_saliency, position)

    for pos, (tok_id, s) in enumerate(zip(input_ids, saliency)):
        tok_id = int(tok_id)
        if tok_id in exclude_special_ids:
            continue

        a = abs(float(s))
        if (tok_id not in best) or (a > best[tok_id][0]):
            best[tok_id] = (a, int(pos))

    if not best:
        return []

    items = [(tok_id, a, pos) for tok_id, (a, pos) in best.items()]
    items.sort(key=lambda x: x[1], reverse=True)
    items = items[:top_k]

    return [{"token_id": tok_id, "saliency": a, "position": pos} for tok_id, a, pos in items]

Fill up row per sentence vectors for top-5 tokens:

---

**Build S1-Vocab**

This function converts the first n text examples into their salience-fingerprints (S1 vectors), stacks those into a matrix, normalizes them, and returns both the matrix and the original texts.

Text example -> Row

<pre>
          column dimension → |V| (the r vector)
        ┌──────────────────────────────────────────────┐
 row 1  │ r(1)[0]   r(1)[1]   r(1)[2]   ...   r(1)[V-1]│
 row 2  │ r(2)[0]   r(2)[1]   r(2)[2]   ...   r(2)[V-1]│
   .    │    .         .         .              .      │
 row N  │ r(N)[0]   r(N)[1]   r(N)[2]   ...   r(N)[V-1]│
        └──────────────────────────────────────────────┘
             ↑
             vocabulary-dimension (S1 vector)***
<pre>

In [29]:
def saliency_to_vocab_vector(input_ids, saliency, vocab_size, top_k=5):
    """
    Implements S1 from the paper: a |V|-dim vector with the
    max |saliency| per vocab item, zeroed except for top-k entries.
    """
    r = np.zeros(vocab_size, dtype=np.float32) # expecting vocab size of bert tokenzier

    for tok_id, s in zip(input_ids, saliency):
        tok_id = int(tok_id)
        r[tok_id] = max(r[tok_id], abs(float(s))) # get max saliency value

    if top_k is not None and top_k < vocab_size:
        # Keep only top-k entries, zero out the rest
        if np.count_nonzero(r) > top_k: # count how many positoons in r are not zero
            top_idx = np.argpartition(r, -top_k)[-top_k:] # Find the indices of the k largest values in r, argsort begin nlogn efficient
            mask = np.zeros_like(r)
            mask[top_idx] = r[top_idx]
            r = mask

    return r


def build_salience_vocab_matrix_from_indices(
    dataset_split,
    indices,
    model,
    tokenizer,
    device,
    top_k=5,
    max_length=256,
):
    """Build a row-normalized vocabulary salience matrix for selected dataset indices.

    For each text, computes token saliency, aggregates the top-k salient tokens into a
    vocabulary-sized vector, and returns the stacked matrix along with the source texts
    and per-example top-k token metadata including raw and normalized saliency scores.
    """
    vocab_size = tokenizer.vocab_size
    R_rows = []
    texts = []
    salient_token_info = [] # save positions

    special_ids = tokenizer.all_special_ids  


    for idx in tqdm(indices):
        text = dataset_split[idx]["text"]
        tokens, sal, ids = grad_l2_saliency(
            text, model, tokenizer, device, max_length=max_length
        )
        r = saliency_to_vocab_vector(ids, sal, vocab_size, top_k=top_k)
        R_rows.append(r)
        texts.append(text)


        # extract token_positions
        topk_vocab = topk_vocab_tokens_with_maxpos(
            ids, sal, top_k=top_k, exclude_special_ids=special_ids
        )
        salient_token_info.append({
            "dataset_idx": idx,
            "text": text,
            "topk_vocab": topk_vocab
        })

    R_vocab = np.stack(R_rows, axis=0)
    # row-wise L2 normalization as in the paper
    norms = np.linalg.norm(R_vocab, axis=1, keepdims=True) + 1e-12
    R_vocab = R_vocab / norms
    #store both raw + normalized saliency per token in salient_token_info
    for row_i, info in enumerate(salient_token_info):
        for item in info["topk_vocab"]:
            tok_id = int(item["token_id"])
            item["saliency_raw"] = float(item["saliency"])              # existing raw abs-saliency
            item["saliency_norm"] = float(R_vocab[row_i, tok_id])       # exact value from r_vocab row

    return R_vocab, texts, salient_token_info

In [30]:
# select indices of negatively predicted examples
neg_indices = [i for i, p in enumerate(test_preds) if p == 0]  

# sample 2500 
# First seed: np.random.seed(42)
# Second seed: 
np.random.seed(1)
# third seed:  np.random.seed(2)
sample_size = min(2500, len(neg_indices))
neg_sample_indices = np.random.choice(neg_indices, size=sample_size, replace=False)

R_vocab_5_neg, texts_neg, salient_token_info_neg = build_salience_vocab_matrix_from_indices(
    test_data,
    neg_sample_indices,
    ft_bert,
    tokenizer,
    device,
    top_k=5,
)


100%|██████████| 2500/2500 [02:31<00:00, 16.53it/s]


Create S1-Vocab for 2500 positively predicted samples

In [31]:
# # select indices of negatively predicted examples
# pos_indices = [i for i, p in enumerate(test_preds) if p == 1]  

# # sample 2500 
# np.random.seed(42)
# sample_size = min(2500, len(pos_indices))
# neg_sample_indices = np.random.choice(pos_indices, size=sample_size, replace=False)

# R_vocab_5_pos, texts_pos, salient_token_info_pos = build_salience_vocab_matrix_from_indices(
#     test_data,
#     neg_sample_indices,
#     ft_bert,
#     tokenizer,
#     device,
#     top_k=5, ######################
# )


## Pattern Mining
Using FP-Growth Algorithm

Imports

Setup Transactions and Lookup Table (Salient tokens per Sentence identified with L2-Grad Saliency Method)

In [32]:
transactions_neg_5 = []
pos_lookup_neg_5 = []   # list of dicts: token_id -> position (for this transaction)

for item in salient_token_info_neg:
    toks = [int(t["token_id"]) for t in item["topk_vocab"]]

    transactions_neg_5.append(toks)
    pos_lookup_neg_5.append({int(t["token_id"]): int(t["position"]) for t in item["topk_vocab"]})

print("num transactions_neg_5:", len(transactions_neg_5))
print("avg len:", np.mean([len(t) for t in transactions_neg_5]))
print(transactions_neg_5[:5])

num transactions_neg_5: 2500
avg len: 5.0
[[11771, 8016, 8467, 19582, 3532], [8810, 10810, 2769, 9364, 5949], [2190, 20113, 2012, 27590, 2991], [4966, 6439, 9364, 28616, 3433], [15258, 2919, 25963, 9996, 5409]]


Running FP-Algorithm

In [33]:
te = TransactionEncoder() # Creates encoder that converts lists of tokens into a binary matrix (One hot encoded)
X = te.fit(transactions_neg_5).transform(transactions_neg_5) # fit and transform bool matrix to prepare for fp-growth

df = pd.DataFrame(X, columns=te.columns_) # to pandas

freq_itemsets_neg_5 = fpgrowth(
    df,
    min_support=0.002, # minimum frequency of items -> 0.002 = 0.2%
    use_colnames=True, # return column names not indices
    max_len=4 # max item size per pattern
)

freq_itemsets_neg_5["length"] = freq_itemsets_neg_5["itemsets"].apply(len) # add length column

def order_itemset_like_sentence(itemset):
    # pick a transaction and use its token positions as the order reference
    items = [int(x) for x in itemset]

    for tx, pm in zip(transactions_neg_5, pos_lookup_neg_5):
        if all(i in pm for i in items):
            return sorted(items, key=lambda i: pm[i])

    return items


freq_itemsets_neg_5["ordered_ids"] = freq_itemsets_neg_5["itemsets"].apply(order_itemset_like_sentence)
freq_itemsets_neg_5["ordered_tokens"] = freq_itemsets_neg_5["ordered_ids"].apply(lambda ids: tokenizer.convert_ids_to_tokens(ids))

Inspecting Frequent, high support, itemsets larger than 1.

In [37]:
freq_itemsets_neg_5[freq_itemsets_neg_5["length"] > 1].sort_values("support", ascending=False).head(40)

,support,itemsets,length,ordered_ids,ordered_tokens
524,0.0104,"(2769, 5949)",2,"[5949, 2769]","[waste, money]"
540,0.0084,"(2732, 1015)",2,"[1015, 2732]","[1, star]"
539,0.0076,"(3737, 3532)",2,"[3532, 3737]","[poor, quality]"
542,0.0060,"(2732, 2028)",2,"[2028, 2732]","[one, star]"
555,0.0056,"(2048, 3340)",2,"[2048, 3340]","[two, stars]"
537,0.0052,"(9364, 1012)",2,"[9364, 1012]","[disappointed, .]"
538,0.0048,"(2025, 2204)",2,"[2025, 2204]","[not, good]"
551,0.0040,"(15640, 9364)",2,"[15640, 9364]","[disappointing, disappointed]"
568,0.0040,"(2769, 3828)",2,"[3828, 2769]","[save, money]"
576,0.0036,"(7962, 8038)",2,"[8038, 7962]","[ya, ##wn]"


### Creating negative R_Vocab with the top 10 salient tokens per sentence.

In [38]:
R_vocab_10_neg, texts_neg_10, salient_token_info_neg_10 = build_salience_vocab_matrix_from_indices(
    test_data,
    neg_sample_indices,
    ft_bert,
    tokenizer,
    device,
    top_k=10, ######################
)

100%|██████████| 2500/2500 [02:05<00:00, 19.93it/s]


Setup Transactions and Lookup Table (Salient tokens per Sentence identified with L2-Grad Saliency Method)

In [39]:
transactions_neg_10 = []
pos_lookup_10 = []   # list of dicts: token_id -> position (for this transaction)

for item in salient_token_info_neg_10:
    toks = [int(t["token_id"]) for t in item["topk_vocab"]]

    transactions_neg_10.append(toks)
    pos_lookup_10.append({int(t["token_id"]): int(t["position"]) for t in item["topk_vocab"]})

print("num transactions_neg_10:", len(transactions_neg_10))
print("avg len:", np.mean([len(t) for t in transactions_neg_10]))
print(transactions_neg_10[:5])

num transactions_neg_10: 2500
avg len: 10.0
[[11771, 8016, 8467, 19582, 3532, 2203, 2757, 3191, 2338, 4454], [8810, 10810, 2769, 9364, 5949, 3688, 2388, 4396, 25946, 2023], [2190, 20113, 2012, 27590, 2991, 3166, 8141, 1996, 6321, 6048], [4966, 6439, 9364, 28616, 3433, 3709, 17099, 2664, 6314, 9159], [15258, 2919, 25963, 9996, 5409, 2769, 2592, 5247, 2172, 4149]]


Running FP-Algorithm

In [40]:
te = TransactionEncoder() # Creates encoder that converts lists of tokens into a binary matrix (One hot encoded)
X = te.fit(transactions_neg_10).transform(transactions_neg_10) # fit and transform bool matrix to prepare for fp-growth

df = pd.DataFrame(X, columns=te.columns_) # to pandas

freq_itemsets_neg_10 = fpgrowth(
    df,
    min_support=0.002, # minimum frequency of items -> 0.002 = 0.2%
    use_colnames=True, # return column names not indices
    max_len=4 # max item size per pattern
)

freq_itemsets_neg_10["length"] = freq_itemsets_neg_10["itemsets"].apply(len) # add length column

def order_itemset_like_sentence(itemset):
    # pick a transaction and use its token positions as the order reference
    items = [int(x) for x in itemset]

    for tx, pm in zip(transactions_neg_10, pos_lookup_10):
        if all(i in pm for i in items):
            return sorted(items, key=lambda i: pm[i])

    return items


freq_itemsets_neg_10["ordered_ids"] = freq_itemsets_neg_10["itemsets"].apply(order_itemset_like_sentence)
freq_itemsets_neg_10["ordered_tokens"] = freq_itemsets_neg_10["ordered_ids"].apply(lambda ids: tokenizer.convert_ids_to_tokens(ids))

Inspecting Frequent, high support, itemsets larger than 1.

In [43]:
freq_itemsets_neg_10[freq_itemsets_neg_10["length"] > 1].sort_values("support", ascending=False).head(40)

,support,itemsets,length,ordered_ids,ordered_tokens
1068,0.0284,"(2769, 5949)",2,"[5949, 2769]","[waste, money]"
1276,0.0148,"(3737, 3532)",2,"[3532, 3737]","[poor, quality]"
1236,0.0140,"(2025, 2204)",2,"[2025, 2204]","[not, good]"
1385,0.0124,"(2769, 2115)",2,"[2115, 2769]","[your, money]"
1522,0.0120,"(2025, 4276)",2,"[2025, 4276]","[not, worth]"
1251,0.0120,"(2025, 1012)",2,"[2025, 1012]","[not, .]"
1062,0.0120,"(9364, 1012)",2,"[1012, 9364]","[., disappointed]"
1555,0.0116,"(2769, 3828)",2,"[3828, 2769]","[save, money]"
1058,0.0112,"(1012, 2023)",2,"[2023, 1012]","[this, .]"
1317,0.0104,"(2732, 1015)",2,"[1015, 2732]","[1, star]"


In [45]:
freq_itemsets_neg_10[freq_itemsets_neg_10["length"] > 2].sort_values("support", ascending=False).head(40)

,support,itemsets,length,ordered_ids,ordered_tokens
1388,0.0064,"(2769, 2115, 5949)",3,"[5949, 2115, 2769]","[waste, your, money]"
1389,0.0044,"(2769, 2115, 3828)",3,"[3828, 2115, 2769]","[save, your, money]"
1073,0.0040,"(2769, 5949, 2023)",3,"[5949, 2769, 2023]","[waste, money, this]"
1525,0.0032,"(2025, 4276, 2769)",3,"[2025, 4276, 2769]","[not, worth, money]"
1591,0.0028,"(8400, 11488, 4487)",3,"[11488, 8400, 4487]","[##ssa, ##point, di]"
1074,0.0024,"(2769, 1012, 5949)",3,"[1012, 5949, 2769]","[., waste, money]"
1560,0.0024,"(2769, 3828, 4965)",3,"[3828, 2769, 4965]","[save, money, buy]"
1378,0.0024,"(2769, 9202, 5949)",3,"[9202, 5949, 2769]","[horrible, waste, money]"
1469,0.0024,"(17634, 8059, 2022)",3,"[17634, 2022, 8059]","[buyer, be, ##ware]"
1479,0.0024,"(2048, 2435, 3340)",3,"[2435, 2048, 3340]","[gave, two, stars]"


---
## Experiment 1: Inspecting Saliency of top 100 high support transactions 

Add Saliency values per transaction